### MD run with OpenMM to resolve protein:ligand clashes in cofolded structures

The restraint tiering used for now:

Backbone at 100 — the fold doesn't move, so your protein coordinates stay essentially identical to what you submit.
Sidechains at 2 — nearly free. This is what actually resolves the clash: the offending sidechain rotates instead of the ligand being pushed.
Ligand at 5 — light. Allows the pose to settle out of the clash while staying anchored near the prediction (and, importantly, not drifting into the empty space where heme used to be).

Two things to validate on before submitting to OpenFold: 
ligand RMSD before/after (you want ≲0.3 Å — if it's moving 1 Å+, the restraints are too weak 
re-run PoseBusters to confirm the clash cleared.

Claude advice:  If the ligand does drift toward the vacant heme site despite LIGAND_K=5, raise it to 10–20, or keep heme in the system as a frozen set of particles using a generic LJ-only treatment — clunkier, but it puts the excluded volume back without needing real heme parameters.

In [ ]:
# OpenMM run SETUP initially generated by Claude Opus 5.5 9/29/2026 and modified manually for Cyp3A4 structure prediction challenge

import numpy as np
import os
import pandas as pd
from pathlib import Path
import openmm as mm
from openmm import app, unit
from openmmforcefields.generators import SystemGenerator
from openff.toolkit.topology import Molecule
from rdkit import Chem
from rdkit.Chem import AllChem
import gemmi
from pdbfixer import PDBFixer
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# ---- config ----
BASE_DIR = Path(os.getenv("submission_output_dir"))
SUBMISSION_DIR = BASE_DIR / "my_structure_submission"
OUT_DIR        = BASE_DIR / "minimized_submission"; OUT_DIR.mkdir(exist_ok=True)
WORK           = BASE_DIR / "min_work"; WORK.mkdir(exist_ok=True)
LIGAND_RESNAME = "LIG"
HEME_RESNAMES  = {"HEM"}
BACKBONE_K     = 100.0   # kcal/mol/A^2 — hold the fold
SIDECHAIN_K    = 2.0     # weak — let pocket sidechains relax away from the ligand
LIGAND_K       = 5.0     # light — allow small pose adjustment, prevent drift
MAX_ITER       = 1000
# ----------------

import gemmi

def _restore_submission_format(minimized_pdb, prot_chain="A", heme_chain="B", lig_chain="C"):
    st = gemmi.read_structure(str(minimized_pdb)); st.setup_entities()
    aa = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

    prot_res, heme_res = [], []
    lig_atoms = []                       # collect atoms from ALL ligand-ish residues
    for ch in st[0]:
        for r in ch:
            n = r.name.strip()
            if n in aa:        prot_res.append(r)
            elif n == "HEM":   heme_res.append(r)
            elif n in ("LIG", "UNK", "UNL"):
                lig_atoms.extend([a for a in r if a.element.name != "H"])   # heavy atoms only

    if not lig_atoms:
        raise ValueError("no ligand atoms found")
    lig = gemmi.Residue(); lig.name = "LIG"; lig.seqid = gemmi.SeqId("1")
    for a in lig_atoms: lig.add_atom(a)

    new = gemmi.Structure(); new.name = st.name
    model = gemmi.Model("1")
    for name, residues in ((prot_chain, prot_res), (heme_chain, heme_res), (lig_chain, [lig])):
        if residues:
            c = gemmi.Chain(name)
            for r in residues: c.add_residue(r)
            model.add_chain(c)
    new.add_model(model)
    new.setup_entities()
    new.write_pdb(str(minimized_pdb))
    
def _fix_protein(prot_pdb, fixed_pdb):
    fixer = PDBFixer(filename=str(prot_pdb))
    fixer.findMissingResidues()
    fixer.missingResidues = {}          # don't build in loops — keep the prediction as-is
    fixer.findMissingAtoms()            # this catches OXT and any missing sidechain atoms
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)
    app.PDBFile.writeFile(fixer.topology, fixer.positions, open(str(fixed_pdb), "w"))
    return fixed_pdb

def _write_subset(src_pdb, dst_pdb, keep_chains):
    """Write a PDB containing only the named chains."""
    st = gemmi.read_structure(str(src_pdb))
    st.setup_entities()
    for model in st:
        for cname in [c.name for c in model if c.name not in keep_chains]:
            model.remove_chain(cname)
    st.write_pdb(str(dst_pdb))


def _reinsert_heme(original_pdb, minimized_pdb, heme_chain="B"):
    """Append the heme (unmoved, from the original) into the minimized structure."""
    orig = gemmi.read_structure(str(original_pdb)); orig.setup_entities()
    mini = gemmi.read_structure(str(minimized_pdb)); mini.setup_entities()

    src_chain = None
    for ch in orig[0]:
        if ch.name == heme_chain:
            src_chain = ch; break
    if src_chain is None:
        raise ValueError(f"chain {heme_chain} not found in {original_pdb}")

    new_ch = gemmi.Chain(heme_chain)
    for res in src_chain:
        new_ch.add_residue(res)
    mini[0].add_chain(new_ch)
    mini.setup_entities()
    mini.write_pdb(str(minimized_pdb))

def minimize_complex(pdb_path, smiles, out_path):
    st = gemmi.read_structure(str(pdb_path)); st.setup_entities()

    # --- split: protein / ligand / heme(set aside) ---
    lig_st, prot_st = st.clone(), st.clone()
    for model in prot_st:
        for ch in list(model):
            for r in [r for r in ch if r.name.strip() in HEME_RESNAMES | {LIGAND_RESNAME}]:
                ch.remove_residue(r) if hasattr(ch, "remove_residue") else None
    # simpler: write protein-only and ligand-only via atom filtering
    prot_pdb = WORK / f"{pdb_path.stem}_prot.pdb"
    lig_pdb  = WORK / f"{pdb_path.stem}_lig.pdb"
    _write_subset(pdb_path, prot_pdb, keep_chains={"A"})    # protein only
    prot_pdb = _fix_protein(prot_pdb, WORK / f"{pdb_path.stem}_prot_fixed.pdb")   # <-- add
    _write_subset(pdb_path, lig_pdb,  keep_chains={"C"})    # compound only (heme excluded)

    # --- ligand as an OpenFF Molecule with correct bond orders from SMILES ---
    ref  = Chem.MolFromSmiles(smiles)                                  # no AddHs — match heavy atoms
    raw  = Chem.MolFromPDBFile(str(lig_pdb), removeHs=True, sanitize=False)
    lig  = AllChem.AssignBondOrdersFromTemplate(ref, raw)
    lig  = Chem.AddHs(lig, addCoords=True)                             # add Hs AFTER matching
    off_mol = Molecule.from_rdkit(lig, allow_undefined_stereo=True)

    # --- build system: amber14 protein + GAFF/OpenFF ligand, implicit-free vacuum ---
    sysgen = SystemGenerator(
        forcefields=["amber/ff14SB.xml"],
        small_molecule_forcefield="gaff-2.11",
        molecules=[off_mol],
        forcefield_kwargs={
            "constraints": app.HBonds,
            "rigidWater": True,
        },
        nonperiodic_forcefield_kwargs={"nonbondedMethod": app.NoCutoff},
    )

    prot = app.PDBFile(str(prot_pdb))
    modeller = app.Modeller(prot.topology, prot.positions)
    modeller.add(off_mol.to_topology().to_openmm(),
                 off_mol.conformers[0].to_openmm())
    modeller.addHydrogens(sysgen.forcefield)
    # what did the ligand end up being called in the assembled topology?
    print({a.residue.name for a in modeller.topology.atoms() if a.residue.name not in
           set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL HOH".split())})

    modeller.topology.setPeriodicBoxVectors(None)        # before create_system

    system = sysgen.create_system(modeller.topology, molecules=[off_mol])
    for f in system.getForces():
        if isinstance(f, mm.NonbondedForce):
            f.setNonbondedMethod(mm.NonbondedForce.NoCutoff)

    # --- positional restraints, tiered ---
    force = mm.CustomExternalForce("k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
    force.addPerParticleParameter("k")
    for p in ("x0","y0","z0"): force.addPerParticleParameter(p)
    pos = modeller.positions
    bb = {"N","CA","C","O"}
    for atom in modeller.topology.atoms():
        if atom.element == app.element.hydrogen: continue
        resn = atom.residue.name.strip()
        if resn in (LIGAND_RESNAME, "UNK", "UNL"):
            k = LIGAND_K
        elif atom.name in bb:
            k = BACKBONE_K
        else:
            k = SIDECHAIN_K
        x, y, z = pos[atom.index].value_in_unit(unit.nanometer)
        force.addParticle(atom.index,
            [k*unit.kilocalories_per_mole/unit.angstrom**2, x*unit.nanometer,
             y*unit.nanometer, z*unit.nanometer])
    system.addForce(force)

    # --- minimize ---
    integrator = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond,
                                             0.002*unit.picoseconds)
    sim = app.Simulation(modeller.topology, system, integrator)
    sim.context.setPositions(modeller.positions)
    e0 = sim.context.getState(getEnergy=True).getPotentialEnergy()
    sim.minimizeEnergy(maxIterations=MAX_ITER)
    state = sim.context.getState(getPositions=True, getEnergy=True)
    e1 = state.getPotentialEnergy()

    # --- write back, re-inserting the untouched heme ---
    app.PDBFile.writeFile(sim.topology, state.getPositions(), open(out_path, "w"))
    _reinsert_heme(pdb_path, out_path, heme_chain="B")
    _restore_submission_format(out_path)
    return e0, e1

In [ ]:
# dev test: minimize the first PDB in the submission directory (saves time in case anything in setup is wrong)
# if this works - proceed with entire set
p = sorted(SUBMISSION_DIR.glob("*.pdb"))[0]
e0, e1 = minimize_complex(p, smiles_of[p.stem], OUT_DIR / p.name)
print(e0, "->", e1)

In [ ]:
# This cell iterates over all PDB files in the submission directory, minimizes them, and records the results.

# ---- SMILES lookup: submission filename stem -> SMILES ----
CHALLENGE_CSV = os.getenv("CHALLENGE_CSV")
ID_COL, SMILES_COL = "id", "SMILES"

df = pd.read_csv(CHALLENGE_CSV)
smiles_of = dict(zip(df[ID_COL].astype(str).str.strip(), df[SMILES_COL]))

print(f"Searching {SUBMISSION_DIR} for PDB files")
pdbs = sorted(SUBMISSION_DIR.glob("*.pdb"))
print(f"{len(pdbs)} structures to minimize")

results = []
for p in pdbs:
    stem = p.stem
    smi = smiles_of.get(stem)
    if smi is None:
        print(f"  [!] {stem}: no SMILES in {CHALLENGE_CSV}; skipping")
        continue
    out = OUT_DIR / p.name
    try:
        e0, e1 = minimize_complex(p, smi, out)
        # ligand displacement check
        a = gemmi.read_structure(str(p));   a.setup_entities()
        b = gemmi.read_structure(str(out)); b.setup_entities()
        def lig_xyz(st):
            return np.array([[at.pos.x, at.pos.y, at.pos.z]
                             for ch in st[0] if ch.name == "C"
                             for r in ch for at in r if at.element.name != "H"])
        xa, xb = lig_xyz(a), lig_xyz(b)
        rmsd = float(np.sqrt(((xa - xb)**2).sum(1).mean())) if xa.shape == xb.shape else np.nan
        print(f"  [+] {stem}: E {e0.value_in_unit(unit.kilocalories_per_mole):.0f} -> "
              f"{e1.value_in_unit(unit.kilocalories_per_mole):.0f} kcal/mol | ligand RMSD {rmsd:.2f} Å")
        results.append(dict(structure=stem, e_before=e0.value_in_unit(unit.kilocalories_per_mole),
                            e_after=e1.value_in_unit(unit.kilocalories_per_mole), lig_rmsd=rmsd))
    except Exception as ex:
        print(f"  [!] {stem}: FAILED {ex}")
        results.append(dict(structure=stem, e_before=np.nan, e_after=np.nan,
                            lig_rmsd=np.nan, error=str(ex)))

res = pd.DataFrame(results)
res.to_csv(BASE_DIR / "minimization_summary.csv", index=False)
print(f"\n{res['lig_rmsd'].notna().sum()}/{len(pdbs)} minimized -> {OUT_DIR}")
print(res.to_string(index=False))

## BACKUP / TEST CELLS

In [ ]:
# dev test minimize two specific problem structures
for stem in ["OCNT-2313038", "OCNT-2313300"]:
    p = Path(SUBMISSION_DIR) / f"{stem}.pdb"
    e0, e1 = minimize_complex(p, smiles_of[stem], Path(OUT_DIR) / p.name)
    print(f"{stem}: {e0} -> {e1}")

In [ ]:
# dev inspect ligand composition in minimized structure
st = gemmi.read_structure(str(Path(OUT_DIR) / "OCNT-2313038.pdb")); st.setup_entities()
for ch in st[0]:
    for r in ch:
        if r.name.strip() == "LIG":
            els = {}
            for a in r: els[a.element.name] = els.get(a.element.name, 0) + 1
            print(f"{len(r)} atoms: {els}")

In [ ]:
# dev look for ligand movements in refined structure
import numpy as np, gemmi

def lig_xyz(path, resnames=("LIG", "UNK", "UNL")):
    st = gemmi.read_structure(str(path)); st.setup_entities()
    return np.array([[a.pos.x, a.pos.y, a.pos.z]
                     for ch in st[0] for r in ch if r.name.strip() in resnames
                     for a in r if a.element.name != "H"])

a, b = lig_xyz(p), lig_xyz(OUT_DIR / p.name)
print("original:", a.shape, " minimized:", b.shape)
if a.shape == b.shape and a.size:
    print("ligand RMSD: %.2f Å" % np.sqrt(((a - b) ** 2).sum(1).mean()))
else:
    print("shape mismatch — atom ordering or count differs")

In [ ]:
# dev look for protein movements in refined structure (backbone and sidechains)
import numpy as np, gemmi

def prot_atoms(path):
    st = gemmi.read_structure(str(path)); st.setup_entities()
    bb, sc = {}, {}
    for ch in st[0]:
        if ch.name != "A": continue
        for r in ch:
            for a in r:
                if a.element.name == "H": continue
                key = (r.seqid.num, a.name)
                (bb if a.name in ("N","CA","C","O") else sc)[key] = [a.pos.x, a.pos.y, a.pos.z]
    return bb, sc

bb0, sc0 = prot_atoms(p)
bb1, sc1 = prot_atoms(OUT_DIR / p.name)

def rmsd(d0, d1):
    keys = sorted(set(d0) & set(d1))
    a = np.array([d0[k] for k in keys]); b = np.array([d1[k] for k in keys])
    return np.sqrt(((a-b)**2).sum(1).mean()), len(keys)

r_bb, n_bb = rmsd(bb0, bb1)
r_sc, n_sc = rmsd(sc0, sc1)
print(f"backbone RMSD: {r_bb:.2f} Å  ({n_bb} atoms)")
print(f"sidechain RMSD: {r_sc:.2f} Å  ({n_sc} atoms)")

# which residues moved most — should be pocket residues near the ligand
per_res = {}
for (num, name), c0 in sc0.items():
    if (num, name) in sc1:
        d = np.linalg.norm(np.array(c0) - np.array(sc1[(num, name)]))
        per_res[num] = max(per_res.get(num, 0), d)
print("\nlargest sidechain displacements:")
for num, d in sorted(per_res.items(), key=lambda x: -x[1])[:10]:
    print(f"  residue {num}: {d:.2f} Å")

In [ ]:
# dev pocket residues within 5 Å of the ligand in the original
st = gemmi.read_structure(str(p)); st.setup_entities()
lig = np.array([[a.pos.x,a.pos.y,a.pos.z] for ch in st[0] if ch.name=="C" for r in ch for a in r])
near = set()
for ch in st[0]:
    if ch.name != "A": continue
    for r in ch:
        for a in r:
            if np.min(np.linalg.norm(lig - np.array([a.pos.x,a.pos.y,a.pos.z]), axis=1)) < 5.0:
                near.add(r.seqid.num); break
print("pocket residues:", sorted(near))

In [ ]:
# dev look for displacements of pocket residues in the refined structure
print("pocket residue displacements:")
for num in sorted([105,108,119,120,215,301,304,305,309,369,370,442,482]):
    if num in per_res:
        print(f"  {num} ({names.get(num)}): {per_res[num]:.2f} Å")

In [ ]:
# dev look for specific protein residues in the original structure
st = gemmi.read_structure(str(p)); st.setup_entities()
names = {r.seqid.num: r.name for ch in st[0] if ch.name=="A" for r in ch}
for num in [205, 88, 123, 266, 237, 96, 486, 254, 234, 476]:
    print(f"  {num}: {names.get(num)}")

In [ ]:
# dev: inspect the minimized structure
target = OUT_DIR / p.name
print("path:", target, "| exists:", target.exists())

st = gemmi.read_structure(str(target)); st.setup_entities()
for ch in st[0]:
    print("chain", ch.name, [(r.name, len(r)) for r in ch][:3], "...")

# raw text check — what the file actually says
import subprocess
print(subprocess.run(["grep", "-c", "LIG", str(target)], capture_output=True, text=True).stdout)
print(subprocess.run(["grep", "LIG", str(target)], capture_output=True, text=True).stdout[:300])

In [ ]:
# test cell for dev

import gemmi
pdb_file = Path(SUBMISSION_DIR / "OCNT-2312554.pdb")
st = gemmi.read_structure(str(pdb_file))
for model in st:
    for ch in model:
        names = {}
        for r in ch:
            names[r.name] = names.get(r.name, 0) + 1
        # show chain, and the non-amino-acid residues in it
        het = {k: v for k, v in names.items() if k not in
               set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())}
        print(f"chain {ch.name}: {len(ch)} residues | het: {het}")

In [ ]:
# dev: inspect the hetero residues in the minimized structure
st = gemmi.read_structure(str(target)); st.setup_entities()
for ch in st[0]:
    for r in ch:
        if r.name.strip() not in set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split()):
            els = {}
            for a in r:
                els[a.element.name] = els.get(a.element.name, 0) + 1
            print(f"chain {ch.name} | {r.name} | {len(r)} atoms | elements: {els}")

In [ ]:
# dev: inspect the hetero residues in the minimized structure
st = gemmi.read_structure(str(OUT_DIR / p.name)); st.setup_entities()
for ch in st[0]:
    resns = {}
    for r in ch:
        resns[r.name] = resns.get(r.name, 0) + 1
    het = {k: v for k, v in resns.items() if k not in
           set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL HOH".split())}
    print(f"chain {ch.name}: {len(ch)} residues | het: {het}")